In [3]:
import numpy as np
import geopandas as gpd
import pandas as pd
import glob
from matplotlib import pyplot as plt
import os
import rioxarray as rio
import xarray as xr
import rasterio
import glob
#from geocube.api.core import make_geocube
from shapely.errors import ShapelyDeprecationWarning
import warnings
import folium
from folium import plugins
warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning) 

ModuleNotFoundError: No module named 'geopandas'

For benchmarking, ideally we would just use all the largefires assosiated with a run. However, for designing and testing a benchmarking system, it's nice to be able to subset to a smaller number of fires, because reading in files take a while. It's also nice when designing test-cases to have a mix of wildfires that do intersect with NIFC perimeters, and some that don't. Unfortunatly, right now it is difficult to query for specific fire qualities without reading in a bunch of files. Here are two ways to deal with that. 


# Technique 1: Reading in all the largefire files. *This technique takes a lot of time up front*

Read in all the fires, sort by fire size, and save a subset of the IDs of the largest fires. Because they are large, they are likely to be the fires that overlap with NIFC perimeters. Then ony query by the IDs of the subset of the biggest fires. 

In [4]:
lf_files = glob.glob('/projects/shared-buckets/gsfc_landslides/FEDSoutput-s3-conus/WesternUS_REDO/2019/Largefire/*')
lf_ids = list(set([file.split('Largefire/')[1].split('_')[0] for file in lf_files])) # unique lf ids

In [5]:
largefire_dict = dict.fromkeys(lf_ids)
for lf_id in lf_ids:
    most_recent_file = [file for file in lf_files if lf_id in file][-1]
    largefire_dict[lf_id] = most_recent_file

In [6]:
def load_large_fire(fireID, year = "2020", path_region = "WesternUS_REDO", layer = "perimeter"):
    '''
    loads in largefire file based on fireID and layer, then preps it for "explore" by adding centriod data. Currently limited to one year. 
    
    INPUTS:
        
        fireID (str): fireID offire of interest. Can be found in gdf files read in by prep_gdf and load_file. Can be selected interactivly form a gdf if use gdf.explore()
        year (str): Year that fires took place. Default to 2019. Availible options differ by path_region. 
        path_region (str): This constructs the path that the fires are stored in. WesternUS and CONUS availible. 
    '''
    lf_files = glob.glob('/projects/shared-buckets/gsfc_landslides/FEDSoutput-s3-conus/' + path_region +'/'+ year +'/Largefire/F' + fireID + '_*') 
    lf_ids = list(set([file.split('Largefire/')[1].split('_')[0] for file in lf_files])) 
    largefire_dict = dict.fromkeys(lf_ids)
    
    for lf_id in lf_ids:
         most_recent_file = [file for file in lf_files if lf_id in file][-1]
         largefire_dict[lf_id] = most_recent_file
    
    gdf = pd.concat([gpd.read_file(file, layer = layer) for key, file in largefire_dict.items()], 
                   ignore_index=True)
    gdf = gdf.to_crs('EPSG:4326')
    gdf['fireID'] = fireID
    gdf['lon'] = gdf.centroid.x
    gdf['lat'] = gdf.centroid.y
    return gdf

In [7]:
frs = []
ids = lf_ids
for i in ids:
    tmp = load_large_fire(i)
    frs.append(tmp)
perimeters = pd.concat(frs) 

ValueError: No objects to concatenate

In [ ]:
max_areas = perimeters.groupby('ID')['farea'].max().sort_values(ascending=False)
print(max_areas.index[0:10]) 

# Technique 2: Read in a shapefile towards the tail end of the season

Snapshot files are easier to read in if you want to compare different fires. Becuase they keep a record of fires for 20 days after the last observation, if you choose a snapshot file late in the season, you frequently get a "snapshot" that includes the biggest fires from the past fire season. The only downside of the snapshot files is that you only get a subset of the fires. If you want a complete piture of all the fires, you need to read in the largefire files. 

fireID's *are* shared between snapshot files and largefire files, so you can use snapshot files to look up fireIDs. 

In [8]:

def load_file(date,layer='perimeter',handle_multi=False,
              only_lf=False,area_lim=5,show_progress=False, year = "2019", path_region = "WesternUS"):
    '''
    loads in snapshot file based on input date and layer
    
    INPUTS:
        
        date (str): string in the form YYYYMMDDAM (or PM)
        layer (str): either "perimeter", "fireline", or "newfirepix"
        handle_multi (bool): drop fire ids in snapshot data that have several polygons
                             associated with them. these are usually several close together
                             static fires that should be filtered out.
        only_lf (bool): only display fires with polygons > area_lim
        area_lim (int): value in km2 to use as lower threshold for largefire filter
        show_progress (bool): print out the file's date once it's loaded.
                              this is helpful when using the function in a loop.
        year (str): Year that fires took place. Default to 2019. Availible options differ by path_region. 
        path_region (str): This constructs the path that the fires are stored in. WesternUS and CONUS availible. 
    
       This function was authored by Eli, and modified by Tess.  
    
    '''
    
    base_path = '/projects/shared-buckets/gsfc_landslides/FEDSoutput-s3-conus/' +path_region+ '/'+ year +'/Snapshot/'
    full_path = os.path.join(base_path,date)
    try: 
        gdf = gpd.read_file(full_path,layer=layer)
        if show_progress:
            print(date,'loaded')
    except: 
        print('could not load file',full_path)
        return None
    
    if handle_multi:
        multi_geoms = gdf.loc[gdf.geometry.geometry.type=='MultiPolygon'].index
        gdf['NumPolygons'] = gdf.loc[multi_geoms,'geometry'].apply(lambda x: len(list(x)))
        too_many_polygons = gdf[gdf['NumPolygons']>4].index
        gdf.drop(too_many_polygons,inplace=True)
    
    if only_lf:
        gdf = gdf[gdf['farea']>area_lim]
    
    return gdf


def prep_gdf(date = '20191031PM',layer='perimeter', handle_multi=True,only_lf=True,area_lim=5, year = "2019", **kwargs):
    '''
    loads in snapshot file based on input date and layer, then preps it for "explore" by adding centriod data and converting datTime files to strings. 
    
    INPUTS:
        
        date (str): string in the form YYYYMMDDAM (or PM)
        layer (str): either "perimeter", "fireline", or "newfirepix"
        handle_multi (bool): drop fire ids in snapshot data that have several polygons
                             associated with them. these are usually several close together
                             static fires that should be filtered out.
        only_lf (bool): only display fires with polygons > area_lim
        area_lim (int): value in km2 to use as lower threshold for largefire filter
        show_progress (bool): print out the file's date once it's loaded.
                              this is helpful when using the function in a loop.
        year (str): Year that fires took place. Default to 2019. Availible options differ by path_region. 
        path_region (str): This constructs the path that the fires are stored in. WesternUS and CONUS availible. 
    '''
    gdf = load_file( date = date, layer = layer,  handle_multi = handle_multi, only_lf = only_lf, area_lim = area_lim, year = year, **kwargs)

### Get explore map to display lat and lon 
    gdf_test = gdf ## Default crs seems to be easting and westing in just US. reproject to lat vs lon

    gdf_test = gdf_test.to_crs('EPSG:4326')
    gdf_test['lon'] = gdf_test.centroid.x
    gdf_test['lat'] = gdf_test.centroid.y

    gdf_test["t"] = gdf_test["t"].astype("str")
# [] dask worker visibility

# []
    gdf_test["t_st"] = gdf_test["t_st"].astype("str")
    gdf_test["t_ed"] = gdf_test["t_ed"].astype("str")
    
    return(gdf_test)

In [10]:
snap = prep_gdf(date = '20190930PM',layer='perimeter',handle_multi=False,only_lf=True,area_lim=4.7, year = "2019", path_region = "WesternUS_REDO")

/tmp/ipykernel_4632/1914022184.py:70: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_test['lon'] = gdf_test.centroid.x
/tmp/ipykernel_4632/1914022184.py:71: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_test['lat'] = gdf_test.centroid.y


In [11]:
snap.explore()

 You can sort snapshot fires by fire qualities, and then use that list of fireID to read in the largefire files, if you need the complete evolution of those fires. 

In [8]:
max_ids_snap = snap.sort_values(by = ["farea"])
print(max_ids_snap[0:10].fireID)

177    11625
722    12443
104    10510
968    12605
185    11515
69      9015
144    10244
465    12393
345    11744
319    12063
Name: fireID, dtype: int64
